# `paper_tmlr_1` n=5 paired confirmation — Colab harness (H100)

Repeats three pilot arms five times each (one seed per run, same hyperparameters as the seed-0 colab pilot) so the headline finding can be upgraded from a single point estimate to a paired comparison with quantified uncertainty.

## Arms

| # | Arm | Trainer | Seeds |
|---|-----|---------|-------|
| 1 | matched-attn baseline    | `train_matched_baseline_scaleup.py` | 0, 1, 2, 3, 4 |
| 2 | Helmholtz Q9d AAAASSSS   | `train_helmholtz_scaleup.py`        | 0, 1, 2, 3, 4 |
| 3 | Hybrid VA (k=4, m=4)     | `train_hybrid_scaleup.py`           | 0, 1, 2, 3, 4 |

## Wall-clock estimates (H100 80 GB)

| Arm | Per-seed | × 5 seeds |
|---|---:|---:|
| matched-attn baseline    | ~5 min  | ~25 min |
| Helmholtz Q9d AAAASSSS   | ~7 min  | ~35 min |
| Hybrid VA (k=4, m=4)     | ~7 min  | ~35 min |
| **Total** |     | **~95 min** |

Per-seed numbers are taken from the corresponding seed-0 pilot wall-clock divided by the H100/A100 throughput ratio (~3×). The notebook is **resume-friendly**: each cell skips already-completed (seed, arm) pairs by checking for the per-seed `*_summary.md` in Drive.

## Decision rule

- **Per-arm**: report mean ± std and 95% CI (Student's t, df = n − 1) of val PPL.
- **Per-arm vs baseline**: paired Δ (per-seed) with mean Δ ± 95% CI.
- **Verdict**: arm wins iff the 95% CI of mean Δ lies entirely below 0.

This upgrades the seed-0 pilot's point estimate to a real paired hypothesis test.


## 1. Environment setup

In [ ]:
import os, sys, subprocess, shutil, json, time
from pathlib import Path

# Set the CUDA allocator config BEFORE any torch import or CUDA context
# creation.  expandable_segments=True helps the caching allocator grow
# and shrink segments to reduce fragmentation.  This is propagated to
# every trainer subprocess we launch via subprocess.Popen, which
# inherits the parent process's environment.
os.environ.setdefault('PYTORCH_ALLOC_CONF', 'expandable_segments:True')
print('PYTORCH_ALLOC_CONF =', os.environ['PYTORCH_ALLOC_CONF'])

IN_COLAB = 'google.colab' in sys.modules
print('In Colab:', IN_COLAB)

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT   = Path('/content/drive/MyDrive/semsimula_pilot')
    REPO_PARENT  = Path('/content')
else:
    DRIVE_ROOT   = Path.home() / 'semsimula_pilot'
    REPO_PARENT  = Path.cwd().parent.parent.parent.parent  # local fallback

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_RESULTS = DRIVE_ROOT / 'results'
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
DRIVE_LOGS    = DRIVE_ROOT / 'logs'
DRIVE_LOGS.mkdir(parents=True, exist_ok=True)
print('Drive root   :', DRIVE_ROOT)
print('Results dir  :', DRIVE_RESULTS)
print('Logs dir     :', DRIVE_LOGS)


In [ ]:
# Clone (or pull) the semsimula repo into Colab's ephemeral disk.
# Replace REPO_URL with your fork or branch as needed.
REPO_URL  = 'https://github.com/dimitarpg13/semsimula-paper.git'
REPO_NAME = 'semsimula'
REPO_DIR  = REPO_PARENT / REPO_NAME

if not REPO_DIR.exists():
    print(f'Cloning {REPO_URL} -> {REPO_DIR} ...')
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
else:
    print(f'{REPO_DIR} already exists; pulling latest ...')
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=False)

SCALEUP_DIR = REPO_DIR / 'notebooks' / 'conservative_arch' / 'scaleup'
assert SCALEUP_DIR.exists(), f'scaleup dir not found at {SCALEUP_DIR}'
print('scaleup dir  :', SCALEUP_DIR)
print('contents     :', sorted(p.name for p in SCALEUP_DIR.iterdir())[:25])


In [ ]:
# Install Python dependencies. Colab already ships with a recent torch+CUDA;
# we only need to top up the smaller helpers.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers', 'datasets', 'pyarrow'], check=True)
print('Dependencies OK')


In [ ]:
# Verify GPU + report device specs.  H100 (80 GB) recommended for the
# tightest wall-clock; A100 (40 GB) and L4 (24 GB) also work but slower.
import torch
print('torch      :', torch.__version__)
print('cuda avail :', torch.cuda.is_available())
if torch.cuda.is_available():
    dev = torch.cuda.get_device_properties(0)
    print(f'GPU         : {dev.name}')
    print(f'Total VRAM  : {dev.total_memory / 1e9:.1f} GB')
    print(f'CUDA cap    : sm_{dev.major}{dev.minor}')
    print(f'CUDA driver : {torch.version.cuda}')
    if 'H100' not in dev.name:
        print('NOTE: wall-clock estimates assume H100; expect ~3x slower on A100, ~5x slower on L4.')
else:
    print('NO CUDA GPU - switch to a GPU runtime: Runtime > Change runtime type > GPU (H100 recommended)')
    raise SystemExit(1)


In [ ]:
# Verify data + logfreq surprisal files are present (tracked in repo).
DATA_DIR     = REPO_DIR / 'notebooks' / 'conservative_arch' / 'data'
LOGFREQ_PATH = SCALEUP_DIR / 'results' / 'logfreq_surprisal_tinystories.npy'
NPZ_PATH     = DATA_DIR / 'tinystories_gpt2_1files_5000000toks.npz'

assert NPZ_PATH.exists(),    f'TinyStories .npz missing at {NPZ_PATH}'
assert LOGFREQ_PATH.exists(), f'logfreq .npy missing at {LOGFREQ_PATH}'
print('TinyStories .npz   :', NPZ_PATH, f'({NPZ_PATH.stat().st_size/1e6:.1f} MB)')
print('logfreq surprisal  :', LOGFREQ_PATH, f'({LOGFREQ_PATH.stat().st_size/1e3:.1f} KB)')


## 2. Run-arm helper (seed-aware skip-if-done)

In [ ]:
import datetime as _dt

def _summary_exists_for(prefix: str, suffix: str = '') -> Path | None:
    """Look for any *_summary.md whose name starts with `prefix` and
    contains `suffix` as a substring. Returns the matching path or None.
    Uses a single-asterisk glob (Python 3.12 rejects adjacent '**' unless
    it is an entire path component) and filters the suffix in Python."""
    for p in sorted(DRIVE_RESULTS.glob(f'{prefix}*_summary.md')):
        if suffix and suffix not in p.name:
            continue
        return p
    return None


def run_arm(name: str, script: str, args: list[str], skip_prefix: str,
            seed: int = 0, log_label: str | None = None,
            extra_skip_suffix: str = '',
            tag_suffix_override: str | None = None) -> int:
    """Launch a single training arm via subprocess.

    Args:
      name              : human-readable name for logging
      script            : trainer script filename inside SCALEUP_DIR
      args              : list of extra CLI args to pass
      skip_prefix       : prefix used to detect a completed run; if any
                          file in DRIVE_RESULTS matches
                          `{skip_prefix}*_summary.md` we skip.
      seed              : seed value (added to args automatically)
      log_label         : optional label for the Drive log file; defaults
                          to `name` lowercased.
      extra_skip_suffix : additional substring required in the matched
                          summary filename (for arms that share a prefix).
      tag_suffix_override : if not None, replaces the default
                          `seed{seed}` tag-suffix string.  Used by the
                          γ-sweep dispatcher to encode γ in the artifact
                          filename.
    """
    label = log_label or name.lower().replace(' ', '_').replace('/', '_')
    existing = _summary_exists_for(skip_prefix, extra_skip_suffix)
    if existing is not None:
        print(f'[run] {name}: SKIP (found {existing.name})')
        return 0
    tag_suffix = tag_suffix_override if tag_suffix_override is not None else f'seed{seed}'
    cmd = [sys.executable, '-u', str(SCALEUP_DIR / script),
           '--mode', 'scaleup',
           '--seed', str(seed),
           '--results-dir', str(DRIVE_RESULTS),
           '--tag-suffix', tag_suffix,
           ] + list(args)
    ts = _dt.datetime.now().strftime('%Y%m%d_%H%M%S')
    log_path = DRIVE_LOGS / f'{label}_{tag_suffix}_{ts}.log'
    print(f'[run] {name}')
    print(f'      cmd: {" ".join(cmd)}')
    print(f'      log: {log_path}')
    t0 = time.time()
    with log_path.open('w') as logf:
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True,
                                bufsize=1)
        for line in proc.stdout:
            print(line, end='')
            logf.write(line)
            logf.flush()
        rc = proc.wait()
    dt = (time.time() - t0) / 3600.0
    print(f'[run] {name}: exit={rc}  elapsed={dt:.2f} h')
    return rc


def run_arm_seed(name: str, script: str, args: list[str],
                 skip_prefix_base: str, seed: int, **kwargs) -> int:
    """Like `run_arm` but `skip_prefix_base` is auto-extended with
    `_seed{seed}` so the skip-if-done check is per-seed (otherwise
    seed-1 would be skipped just because seed-0 already wrote a
    summary file with the same prefix)."""
    return run_arm(
        name=f'{name} seed={seed}',
        script=script,
        args=args,
        skip_prefix=f'{skip_prefix_base}_seed{seed}',
        seed=seed,
        **kwargs,
    )


## 3. n=5 paired confirmation

Each cell loops over seeds 0..4 and skips already-completed runs.  Run them in any order; the aggregator only needs the per-seed `*_summary.md` and `*_ckpt_latest.pt` files to exist.

If a Colab session disconnects mid-loop, simply re-run the cells: the skip-if-done logic picks up where you left off.


In [ ]:
SEEDS = [0, 1, 2, 3, 4]
print('seeds:', SEEDS)


In [ ]:
# Arm 1 (n=5) - matched-attention baseline (~25 min total on H100)
for seed in SEEDS:
    rc = run_arm_seed(
        name='matched-attn baseline',
        script='train_matched_baseline_scaleup.py',
        args=[],
        skip_prefix_base='matched_baseline_scaleup_scaleup',
        seed=seed,
    )
    assert rc == 0, f'matched-attn seed={seed} returned exit {rc}'


In [ ]:
# Arm 2 (n=5) - Helmholtz Q9d AAAASSSS (~35 min total on H100)
for seed in SEEDS:
    rc = run_arm_seed(
        name='Helmholtz Q9d AAAASSSS',
        script='train_helmholtz_scaleup.py',
        args=['--schedule', 'AAAASSSS'],
        skip_prefix_base='helmholtz_AAAASSSS_L8_scaleup_scaleup',
        seed=seed,
    )
    assert rc == 0, f'Helmholtz Q9d seed={seed} returned exit {rc}'


In [ ]:
# Arm 3 (n=5) - Hybrid VA (k=4, m=4) (~35 min total on H100)
for seed in SEEDS:
    rc = run_arm_seed(
        name='Hybrid VA k=4 m=4',
        script='train_hybrid_scaleup.py',
        args=['--n-attn', '4', '--n-splm', '4'],
        skip_prefix_base='hybrid_VA_k4_m4_scaleup_scaleup',
        seed=seed,
    )
    assert rc == 0, f'Hybrid VA seed={seed} returned exit {rc}'


## 4. Aggregate n=5 statistics

Reads all per-seed checkpoints in `DRIVE_RESULTS`, computes per-arm mean ± std + 95% CI of val PPL, and produces the **paired Δ table** (per-seed Δ vs matched-attention baseline + 95% CI from Student's t with df = n − 1).

Outputs:
- `PILOT_N5_RESULTS.md`  — per-arm marginal table, paired-Δ table, per-seed transparency table, generalization-gap table
- `n5_paired_strip.png`  — left panel: absolute val PPL strip plot per arm; right panel: paired Δ vs matched-attn baseline


In [ ]:
seed_csv = ','.join(str(s) for s in SEEDS)
subprocess.run([sys.executable, str(SCALEUP_DIR / 'aggregate_n5_results.py'),
                '--results-dir', str(DRIVE_RESULTS),
                '--seeds', seed_csv,
                '--out-dir', str(DRIVE_RESULTS)],
               check=True)
print('\n--- PILOT_N5_RESULTS.md ---\n')
print((DRIVE_RESULTS / 'PILOT_N5_RESULTS.md').read_text())


In [ ]:
from IPython.display import Image, display
p = DRIVE_RESULTS / 'n5_paired_strip.png'
if p.exists():
    print(p)
    display(Image(str(p)))
else:
    print('n5_paired_strip.png missing - re-run aggregator')


## 5. Persistence & next steps

All artifacts live at:

```
/content/drive/MyDrive/semsimula_pilot/
├── results/
│   ├── matched_baseline_scaleup_scaleup_seed{0..4}_*
│   ├── helmholtz_AAAASSSS_L8_scaleup_scaleup_seed{0..4}_*
│   ├── hybrid_VA_k4_m4_scaleup_scaleup_seed{0..4}_*
│   ├── PILOT_N5_RESULTS.md
│   └── n5_paired_strip.png
└── logs/
    └── <arm>_seed{0..4}_<timestamp>.log
```

### What to do with the results

1. Download `PILOT_N5_RESULTS.md` and `n5_paired_strip.png` from Drive to your laptop.
2. Place them under `notebooks/conservative_arch/scaleup/results/n5_confirmation/` in the local repo.
3. Commit the markdown + plot (skip `.pt` checkpoints — they are large and not needed for the paper).
4. Update `paper_tmlr_1` Discussion §10 with the n=5 mean ± 95% CI numbers, replacing any seed-0 point estimates.
5. **Verdict logic**: if the paired Δ vs matched-attn for Helmholtz Q9d (and/or Hybrid VA) has a 95% CI that lies entirely below zero, the Pareto-frontier claim is statistically confirmed; otherwise report the tie with the CI brackets.

### Re-running on a different GPU

This notebook works on A100/L4 too — just expect ~3× wall-clock vs the H100 estimates above. No code changes needed.

### Adding more seeds

Append new seed numbers to `SEEDS` in the dispatch cell (e.g. `[0, 1, 2, 3, 4, 5, 6, 7]`) and re-run all cells.  The aggregator's CLI flag `--seeds 0,1,2,3,4,5,6,7` will pick them all up automatically.
